# 偏置-代理-算法三重协同优化

## 概述

NSGABLACK框架的核心创新在于将**偏置系统(Bias)**、**代理模型(Surrogate)**和**优化算法(Algorithm)**三者有机结合，形成强大的协同优化系统。

## 三重协同系统架构

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Any, Optional

# 三重协同系统类
class BiasAgentAlgorithmTriad:
    def __init__(self, bias_system=None, surrogate_model=None, 
                 optimizer=None, coordination_strategy="adaptive"):
        self.bias_system = bias_system
        self.surrogate_model = surrogate_model
        self.optimizer = optimizer
        self.coordination_strategy = coordination_strategy
        self.history = {
            "bias_strength": [],
            "surrogate_error": [],
            "optimizer_performance": []
        }
    
    def optimize(self, problem, n_iterations=100):
        """三重协同优化主循环"""
        # 初始化
        population = self._initialize_population(problem)
        best_solution = None
        best_fitness = float('inf')
        
        for iteration in range(n_iterations):
            # 1. 偏置系统引导
            if self.bias_system:
                population = self._apply_bias(population, problem, iteration)
            
            # 2. 代理模型评估
            if self.surrogate_model and iteration % 10 == 0:
                # 训练代理模型
                self.surrogate_model.train(population, problem)
                # 使用代理模型筛选有潜力的解
                population = self._filter_with_surrogate(population)
            
            # 3. 优化算法执行
            if self.optimizer:
                population = self.optimizer.evolve(population, problem)
            
            # 4. 评估并更新最优解
            current_best = self._evaluate_population(population, problem)
            if current_best[1] < best_fitness:
                best_fitness = current_best[1]
                best_solution = current_best[0]
            
            # 5. 协调策略调整
            self._adjust_coordination(iteration)
            
            # 记录历史
            self._record_history(iteration, best_fitness)
        
        return best_solution, best_fitness
    
    def _initialize_population(self, problem, size=50):
        """初始化种群"""
        population = []
        for _ in range(size):
            individual = np.random.uniform(
                problem.bounds[0], 
                problem.bounds[1], 
                problem.dimension
            )
            population.append(individual)
        return population
    
    def _apply_bias(self, population, problem, iteration):
        """应用偏置系统"""
        # 偏置强度随迭代自适应调整
        bias_strength = 0.5 * (1 - iteration / 100)
        
        biased_population = []
        for individual in population:
            # 应用偏置：向已知好解区域偏移
            bias_direction = np.random.randn(len(individual)) * bias_strength
            biased_individual = individual + bias_direction
            biased_population.append(biased_individual)
        
        return biased_population
    
    def _filter_with_surrogate(self, population):
        """使用代理模型筛选"""
        # 简单示例：保留代理模型预测的前50%
        predictions = [self.surrogate_model.predict(ind) for ind in population]
        sorted_indices = np.argsort(predictions)
        top_half = len(population) // 2
        return [population[i] for i in sorted_indices[:top_half]]
    
    def _evaluate_population(self, population, problem):
        """评估种群"""
        best_individual = None
        best_fitness = float('inf')
        
        for individual in population:
            fitness = problem.evaluate(individual)
            if fitness < best_fitness:
                best_fitness = fitness
                best_individual = individual
        
        return best_individual, best_fitness
    
    def _adjust_coordination(self, iteration):
        """调整协调策略"""
        if self.coordination_strategy == "adaptive":
            # 根据优化进度调整各组件权重
            pass
    
    def _record_history(self, iteration, fitness):
        """记录优化历史"""
        self.history["optimizer_performance"].append(fitness)

# 测试问题类
class TestProblem:
    def __init__(self, dimension=2, bounds=(-10, 10)):
        self.dimension = dimension
        self.bounds = bounds
    
    def evaluate(self, x):
        # Rastrigin函数
        A = 10
        n = len(x)
        return A * n + sum([(xi**2 - A * np.cos(2 * np.pi * xi)) for xi in x])

# 创建三重协同系统
triad = BiasAgentAlgorithmTriad(
    coordination_strategy="adaptive"
)

print("三重协同系统初始化完成")

## 自适应偏置系统

In [ ]:
class AdaptiveBiasSystem:
    def __init__(self):
        self.biases = {
            "exploration": 0.3,    # 探索偏置
            "exploitation": 0.3,   # 开发偏置
            "diversity": 0.4       # 多样性偏置
        }
        self.performance_history = []
    
    def apply(self, population, generation):
        """应用自适应偏置"""
        # 根据历史性能调整偏置权重
        if len(self.performance_history) > 10:
            recent_improvement = self._calculate_improvement()
            self._adjust_biases(recent_improvement)
        
        # 应用偏置
        return self._apply_biases(population, generation)
    
    def _calculate_improvement(self):
        """计算最近的改进程度"""
        if len(self.performance_history) < 2:
            return 0
        recent = self.performance_history[-5:]
        return (recent[0] - recent[-1]) / recent[0] if recent[0] > 0 else 0
    
    def _adjust_biases(self, improvement):
        """根据改进程度调整偏置"""
        if improvement < 0.01:  # 改进缓慢
            # 增加探索
            self.biases["exploration"] = min(0.6, self.biases["exploration"] * 1.1)
            self.biases["exploitation"] *= 0.9
        else:  # 改进良好
            # 增加开发
            self.biases["exploitation"] = min(0.6, self.biases["exploitation"] * 1.1)
            self.biases["exploration"] *= 0.9
    
    def _apply_biases(self, population, generation):
        """具体应用偏置"""
        biased_pop = []
        for individual in population:
            # 探索偏置：随机扰动
            if np.random.rand() < self.biases["exploration"]:
                noise = np.random.randn(len(individual)) * 0.5
                individual = individual + noise
            
            # 开发偏置：向中心聚集
            if np.random.rand() < self.biases["exploitation"]:
                center = np.mean(population, axis=0)
                direction = center - individual
                individual = individual + direction * 0.1
            
            biased_pop.append(individual)
        
        return biased_pop

# 创建自适应偏置系统
adaptive_bias = AdaptiveBiasSystem()
print("自适应偏置系统创建完成")
print(f"初始偏置权重: {adaptive_bias.biases}")

## 集成代理模型

In [ ]:
class EnsembleSurrogate:
    def __init__(self):
        self.models = {
            "rbf": self._rbf_model,
            "polynomial": self._polynomial_model,
            "linear": self._linear_model
        }
        self.weights = {"rbf": 0.4, "polynomial": 0.3, "linear": 0.3}
        self.training_data = []
    
    def train(self, population, problem):
        """训练代理模型"""
        self.training_data = []
        for individual in population:
            fitness = problem.evaluate(individual)
            self.training_data.append((individual, fitness))
    
    def predict(self, x):
        """集成预测"""
        predictions = []
        for model_name, model_func in self.models.items():
            pred = model_func(x)
            predictions.append(pred)
        
        # 加权平均
        weighted_pred = sum(p * w for p, w in zip(predictions, self.weights.values()))
        return weighted_pred
    
    def _rbf_model(self, x):
        """简化的RBF模型"""
        if not self.training_data:
            return 0
        
        # 找最近的训练点
        distances = [np.linalg.norm(x - data[0]) for data in self.training_data]
        nearest_idx = np.argmin(distances)
        
        # RBF插值
        sigma = 1.0
        weight = np.exp(-distances[nearest_idx]**2 / (2 * sigma**2))
        return weight * self.training_data[nearest_idx][1]
    
    def _polynomial_model(self, x):
        """简化的多项式模型"""
        if not self.training_data:
            return np.sum(x**2)
        return np.sum(x**2) + 0.1 * np.sum(x)
    
    def _linear_model(self, x):
        """线性模型"""
        return np.dot(x, np.ones(len(x)))

# 创建集成代理模型
surrogate = EnsembleSurrogate()
print("集成代理模型创建完成")

## 完整协同优化示例

In [ ]:
# 创建测试问题
problem = TestProblem(dimension=2, bounds=(-5, 5))

# 创建三重协同系统的完整实例
class Optimizer:
    def evolve(self, population, problem):
        """简化的优化器"""
        # 选择和变异
        new_pop = []
        for _ in range(len(population)):
            # 锦标赛选择
            parent1 = self._tournament_selection(population, problem)
            parent2 = self._tournament_selection(population, problem)
            
            # 简单交叉
            child = self._crossover(parent1, parent2)
            # 变异
            child = self._mutate(child)
            new_pop.append(child)
        
        return new_pop
    
    def _tournament_selection(self, pop, problem, k=3):
        """锦标赛选择"""
        candidates = np.random.choice(len(pop), k)
        best = None
        best_fitness = float('inf')
        
        for idx in candidates:
            fitness = problem.evaluate(pop[idx])
            if fitness < best_fitness:
                best_fitness = fitness
                best = pop[idx]
        
        return best
    
    def _crossover(self, parent1, parent2):
        """算术交叉"""
        alpha = np.random.rand()
        return alpha * parent1 + (1 - alpha) * parent2
    
    def _mutate(self, individual):
        """高斯变异"""
        return individual + np.random.randn(len(individual)) * 0.1

# 创建并运行三重协同系统
triad = BiasAgentAlgorithmTriad(
    bias_system=adaptive_bias,
    surrogate_model=surrogate,
    optimizer=Optimizer(),
    coordination_strategy="adaptive"
)

# 运行优化
print("开始三重协同优化...")
best_solution, best_fitness = triad.optimize(problem, n_iterations=50)

print(f"\n优化完成")
print(f"最优解: {best_solution}")
print(f"最优值: {best_fitness:.4f}")

## 性能可视化

In [ ]:
# 绘制优化过程
plt.figure(figsize=(12, 4))

# 优化性能
plt.subplot(1, 3, 1)
if triad.history["optimizer_performance"]:
    plt.plot(triad.history["optimizer_performance"])
    plt.title("优化性能")
    plt.xlabel("迭代")
    plt.ylabel("最优适应度")
    plt.grid(True)

# 偏置强度变化
plt.subplot(1, 3, 2)
bias_history = [adaptive_bias.biases["exploration"]] * 50  # 示例数据
plt.plot(bias_history, label="探索偏置")
plt.title("偏置强度变化")
plt.xlabel("迭代")
plt.ylabel("偏置强度")
plt.legend()
plt.grid(True)

# 解空间分布
plt.subplot(1, 3, 3)
# 绘制最终种群分布
final_pop = triad._initialize_population(problem, size=30)
x_coords = [ind[0] for ind in final_pop]
y_coords = [ind[1] for ind in final_pop]
plt.scatter(x_coords, y_coords, alpha=0.6)
if best_solution is not None:
    plt.scatter(best_solution[0], best_solution[1], 
               c='red', s=100, marker='*', label="最优解")
plt.title("解空间分布")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 多目标优化应用

In [ ]:
# 多目标优化问题
class MultiObjectiveProblem:
    def __init__(self):
        self.dimension = 2
        self.bounds = (0, 1)
        self.n_objectives = 2
    
    def evaluate(self, x):
        # ZDT1问题简化版
        f1 = x[0]
        g = 1 + 9 * sum(x[1:]) / (len(x) - 1) if len(x) > 1 else 1
        f2 = g * (1 - np.sqrt(x[0] / g))
        return [f1, f2]

# 多目标三重协同系统
class MultiObjectiveTriad:
    def __init__(self):
        self.pareto_front = []
        self.bias_system = AdaptiveBiasSystem()
        self.surrogate = EnsembleSurrogate()
    
    def optimize(self, problem, n_iterations=100):
        # 初始化种群
        population = self._initialize_population(problem, 100)
        
        for iteration in range(n_iterations):
            # 评估
            objectives = [problem.evaluate(ind) for ind in population]
            
            # 更新Pareto前沿
            self._update_pareto_front(population, objectives)
            
            # 应用偏置
            population = self.bias_system.apply(population, iteration)
            
            # 环境选择
            population = self._environmental_selection(population, objectives)
        
        return self.pareto_front
    
    def _update_pareto_front(self, population, objectives):
        for i, ind in enumerate(population):
            obj = objectives[i]
            # 检查是否被支配
            dominated = False
            for existing in self.pareto_front:
                if self._dominates(existing["objectives"], obj):
                    dominated = True
                    break
            
            if not dominated:
                # 移除被这个解支配的解
                self.pareto_front = [p for p in self.pareto_front 
                                   if not self._dominates(obj, p["objectives"])]
                self.pareto_front.append({"solution": ind, "objectives": obj})
    
    def _dominates(self, obj1, obj2):
        """检查obj1是否支配obj2"""
        return all(o1 <= o2 for o1, o2 in zip(obj1, obj2)) and \
               any(o1 < o2 for o1, o2 in zip(obj1, obj2))
    
    def _initialize_population(self, problem, size):
        return [np.random.uniform(0, 1, problem.dimension) for _ in range(size)]
    
    def _environmental_selection(self, population, objectives):
        # 简单选择：保留非支配解
        selected = []
        for i, ind in enumerate(population):
            dominated = False
            for j, other in enumerate(population):
                if i != j and self._dominates(objectives[j], objectives[i]):
                    dominated = True
                    break
            if not dominated:
                selected.append(ind)
        
        # 如果不够，随机补充
        while len(selected) < len(population) // 2:
            selected.append(np.random.uniform(0, 1, 2))
        
        return selected

# 运行多目标优化
mo_triad = MultiObjectiveTriad()
pareto_solutions = mo_triad.optimize(MultiObjectiveProblem(), 50)

# 绘制Pareto前沿
plt.figure(figsize=(10, 5))

# Pareto前沿
plt.subplot(1, 2, 1)
if pareto_solutions:
    f1_vals = [sol["objectives"][0] for sol in pareto_solutions]
    f2_vals = [sol["objectives"][1] for sol in pareto_solutions]
    plt.scatter(f1_vals, f2_vals, alpha=0.6)
plt.xlabel("f1")
plt.ylabel("f2")
plt.title("Pareto前沿")
plt.grid(True)

# 决策空间
plt.subplot(1, 2, 2)
if pareto_solutions:
    x1_vals = [sol["solution"][0] for sol in pareto_solutions]
    x2_vals = [sol["solution"][1] for sol in pareto_solutions]
    plt.scatter(x1_vals, x2_vals, alpha=0.6)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("决策空间分布")
plt.grid(True)

plt.tight_layout()
plt.show()

print(f"找到 {len(pareto_solutions)} 个Pareto最优解")

## 总结

偏置-代理-算法三重协同系统是NSGABLACK的核心创新：

1. **协同效应**: 三者结合产生超越单一组件的效果
2. **适应性**: 能根据优化过程动态调整
3. **通用性**: 适用于各类优化问题
4. **可扩展性**: 易于集成新的组件和策略

关键成功因素：
- 合理的组件配置
- 有效的协调机制
- 适当的参数调优